# 🔎 Delhi Deep Dive — Module 07: Risk Lookup Function
## Query any Delhi district's complete risk profile

In [ ]:
import pandas as pd
import os
import sys

OUTPUT_DIR = '../data/outputs/delhi'

DELHI_WEIGHTS = {
    'drought_risk_score': 0.15,
    'flood_risk_score': 0.25,
    'heatwave_risk_score': 0.20,
    'airquality_risk_score': 0.25,
    'waterscarcity_risk_score': 0.15,
}

YAMUNA_PROXIMITY = {
    'North East Delhi': 1.0, 'East Delhi': 0.9, 'Shahdara': 0.9,
    'North Delhi': 0.7, 'Central Delhi': 0.5, 'New Delhi': 0.3,
    'South East Delhi': 0.4, 'South Delhi': 0.2, 'West Delhi': 0.1,
    'South West Delhi': 0.1, 'North West Delhi': 0.2,
}

GROUNDWATER_STRESS = {
    'South Delhi': 1.0, 'South West Delhi': 0.95, 'West Delhi': 0.90,
    'North West Delhi': 0.85, 'Central Delhi': 0.80, 'New Delhi': 0.75,
    'South East Delhi': 0.85, 'North Delhi': 0.60, 'East Delhi': 0.55,
    'Shahdara': 0.50, 'North East Delhi': 0.45,
}

comp = pd.read_csv(os.path.join(OUTPUT_DIR, 'delhi_composite_scores.csv'))
comp['_key'] = comp['district'].str.strip().str.lower()

def get_delhi_risk_profile(district: str, year: int) -> dict:
    """
    Returns the complete risk profile for a Delhi district and year.
    
    Parameters
    ----------
    district : str  — e.g. 'North East Delhi' (case-insensitive)
    year     : int  — 2010 to 2023
    
    Returns
    -------
    dict with all scores, category, recommendation, and flags
    """
    row = comp[(comp['_key'] == district.strip().lower()) & (comp['year'] == year)]
    if row.empty:
        available = sorted(comp['district'].unique())
        return {'error': f'No data for {district!r} in {year}. Available: {available}'}
    row = row.iloc[0]

    scores = {
        'Drought':        float(row.get('drought_risk_score',       0) or 0),
        'Flood':          float(row.get('flood_risk_score',         0) or 0),
        'Heatwave':       float(row.get('heatwave_risk_score',      0) or 0),
        'Air Quality':    float(row.get('airquality_risk_score',    0) or 0),
        'Water Scarcity': float(row.get('waterscarcity_risk_score', 0) or 0),
    }
    sorted_scores = sorted(scores, key=scores.get, reverse=True)
    primary, secondary = sorted_scores[0], sorted_scores[1]

    cat = str(row['delhi_composite_category'])
    rec_map = {
        'LOW':       'Standard processing. Annual climate review recommended.',
        'MEDIUM':    'Climate addendum required. Review flood plain and AQI zone.',
        'HIGH':      'Climate risk insurance mandatory. Flag for environmental due diligence.',
        'VERY HIGH': 'Do not approve without independent climate assessment and insurance.',
    }
    recommendation = rec_map.get(cat, '') + f' Primary risks: {primary} and {secondary}.'

    gw_stress = GROUNDWATER_STRESS.get(district, 0.5)
    gw_label  = ('CRITICAL' if gw_stress >= 0.9 else
                 'HIGH'     if gw_stress >= 0.7 else
                 'MODERATE' if gw_stress >= 0.4 else 'LOW')

    return {
        'district':                 district,
        'year':                     year,
        'drought_risk_score':       round(scores['Drought'], 2),
        'flood_risk_score':         round(scores['Flood'], 2),
        'heatwave_risk_score':      round(scores['Heatwave'], 2),
        'airquality_risk_score':    round(scores['Air Quality'], 2),
        'waterscarcity_risk_score': round(scores['Water Scarcity'], 2),
        'delhi_composite_score':    float(row['delhi_composite_score']),
        'delhi_composite_category': cat,
        'primary_risk':             primary,
        'secondary_risk':           secondary,
        'bank_recommendation':      recommendation,
        'yamuna_floodplain':        YAMUNA_PROXIMITY.get(district, 0) >= 0.7,
        'groundwater_stress':       gw_label,
    }

print('✓ Lookup function loaded. Districts available:')
print(sorted(comp['district'].unique()))

## Example Queries

In [ ]:
import json

# Example 1: Yamuna floodplain area
profile1 = get_delhi_risk_profile('North East Delhi', 2023)
print('=== North East Delhi (2023) ===')
print(json.dumps(profile1, indent=2))

In [ ]:
# Example 2: Groundwater-stressed south
profile2 = get_delhi_risk_profile('South Delhi', 2023)
print('=== South Delhi (2023) ===')
print(json.dumps(profile2, indent=2))

In [ ]:
# Example 3: All districts for 2023 — ranked
results = []
for d in sorted(comp['district'].unique()):
    p = get_delhi_risk_profile(d, 2023)
    if 'error' not in p:
        results.append(p)

ranked = pd.DataFrame(results)[['district','delhi_composite_score','delhi_composite_category','primary_risk','groundwater_stress','yamuna_floodplain']]
ranked = ranked.sort_values('delhi_composite_score', ascending=False).reset_index(drop=True)
ranked.index += 1
print('=== ALL DELHI DISTRICTS — 2023 RISK RANKING ===')
print(ranked.to_string())

## Usage

```python
# To use this lookup from any notebook or script:
from delhi.lookup import get_delhi_risk_profile  # if packaged

# Or paste the function directly.
profile = get_delhi_risk_profile('West Delhi', 2022)
print(profile['bank_recommendation'])
```

**This is the final module in the Delhi Deep Dive pipeline.**